In [1]:
import pandas


ModuleNotFoundError: No module named 'pandas'

In [ ]:
#read in eval_set batches from gt_all_eval.txt

#for each batch

#for each image

#run detect

#if no_of_results = 0: print zero row

#for each result:

#run htr_model

#print result row, and include confidence and gt

In [1]:
import pandas as pd
from glob import glob
from pathlib import Path

import torch, torchvision
print(torch.__version__, torch.cuda.is_available())

# Check MMDetection installation
import mmdet
print(mmdet.__version__)

# Check mmcv installation
from mmcv.ops import get_compiling_cuda_version, get_compiler_version
print(get_compiling_cuda_version())
print(get_compiler_version())

import mmcv
import matplotlib.pyplot as plt

1.10.0 True
2.18.0
10.2
GCC 7.3


In [2]:
from mmcv import Config
cfg = Config.fromfile('/home/erik/Riksarkivet/Projects/fastighet/models/configs/cascade_rcnn2.py')

In [3]:
from mmdet.apis import inference_detector, init_detector, show_result_pyplot
checkpoint = '/home/erik/Riksarkivet/Projects/fastighet/models/cascade_rcnn_1/latest.pth'
model = init_detector(cfg, checkpoint, device='cuda:0')

Use load_from_local loader


In [4]:

df = pd.read_csv('//home/erik/Riksarkivet/Projects/fastighet/data/htr_training_set_1_million/gt_all_eval.txt', delimiter='|', names=['path', 'gt'])

batches = []

for index, row in df.iterrows():
    batch = row.path.split('/')[0]
    if batch not in batches:
        batches.append(batch)

batches.sort()

In [5]:
df_million_single = pd.read_csv('/home/erik/Riksarkivet/Projects/fastighet/data/miljonsetet/miljon_testet_facit_enkel_FNR.txt', delimiter='\t', index_col=2, encoding='latin-1', dtype=str)
df_million_multiple = pd.read_csv('/home/erik/Riksarkivet/Projects/fastighet/data/miljonsetet/miljon_testet_facit_flera_FNR_datum_stigande.txt', index_col=2, delimiter='\t', encoding='latin-1', dtype=str)
df_million_conc = pd.concat([df_million_single, df_million_multiple])
#remove duplicates

In [10]:
df_page = df_million_multiple.loc['10001009_00000027']
print(len(df_page))

3


In [4]:
df_test = df_million_conc.loc['100010301_00000279']



KeyError: '100010301_00000279'

In [16]:
%%time

#write fnr with gts

from glob import glob
import os
from pathlib import Path

eval_dict = {}

path_to_batches = '/home/erik/Riksarkivet/Projects/fastighet/data/miljonsetet/all_batches'

for batch in batches:


    not_indexed = 0


    imgs = glob(os.path.join(path_to_batches, batch, '**'))

    imgs.sort()

    for img in imgs:

        gts = []
        img_name = Path(img).name.split('.')[0]
        
        
        try:
            rows = df_million_conc.loc[img_name]
        except:
            not_in_df_million_total += 1
            not_in_df_batch += 1
            eval_dict[img_name] = None
            continue
        
        eval_dict[img_name] = []
        
        #if one result
        if isinstance(rows, pd.Series):
            
            trakt = rows['GTRAKT']
            block = rows['GBLOCK']
            enhet = rows['GENHET']
            socken = rows['GKOMMUN_SHORT']
            gts.append(socken + ';' + trakt + ';' + block + ';' + enhet)
            
        #if several results
        else:
            for i, (index, row) in enumerate(rows.iterrows()):

                trakt = row['GTRAKT']
                block = row['GBLOCK']
                enhet = row['GENHET']
                socken = row['GKOMMUN_SHORT']

                gts.append(socken + ';' + trakt + ';' + block + ';' + enhet)
        
        #print(gts)

        im = mmcv.imread(img)
        result = inference_detector(model, im)

        if len(result[0]) == 0:
            pass
        
        elif len(result[0]) == 1:
            res_dict = {}
            res_dict['gts'] = gts
            res_dict['det_prob'] = result[0][0][4]
            eval_dict[img_name].append(res_dict)
            #print(result)
            #out_path = os.path.join(batch_out_dir, Path(img).name)
            #cropped_img = mmcv.imcrop(im, result[0][0][0:4])
            #mmcv.imwrite(cropped_img, out_path)
            
            #img_path = os.path.join(Path(batch).name, Path(img).name)
            #f.write(img_path + '|' + trakt + ';' + block + ';' + enhet + '\n')
        
        elif len(result[0]) > 1:
            for i, res in enumerate(result[0]):
                res_dict = {}
                res_dict['gts'] = gts
                res_dict['det_prob'] = res[4]
                eval_dict[img_name].append(res_dict)



       

    #print(batch + ':' + str(not_indexed))
    break



IndexError: invalid index to scalar variable.

In [7]:
print(len(rows))

18
